In [ ]:
# ============================================================
# Team Asha - 04_model_seeds
# Multi-seed ablation. Reuses the committed preprocessing output:
# NO window extraction, NO daily profiler, NO 64 GB read.
#
# Add TWO inputs to this notebook:
#   1. Notebook Output of the fog-preprocessing commit  -> set PRE below
#   2. The TLVMC competition dataset (for subjects.csv)
# ============================================================
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, f1_score

# ---- EDIT THIS: the mounted path of your committed notebook output
PRE = next(p for p in Path("/kaggle/input").iterdir()
           if (p/"windows_meta.parquet").exists())
ROOT = next(p for p in [
    Path("/kaggle/input/competitions/tlvmc-parkinsons-freezing-gait-prediction"),
    Path("/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction")] if p.exists())
OUT = Path("/kaggle/working")
print("PRE :", PRE)
print("ROOT:", ROOT)

FS, WIN_S, STRIDE_S, BAND_HALF_S = 100, 4.0, 0.5, 0.25
WIN_N = int(WIN_S*FS); N_FOLDS = 6
HORIZONS = [0.5,1.0,1.5,2.0,2.5,3.0,4.0,5.0]
FEAT_NAMES = ["mag_mean","mag_std","mag_absdev","jerk_rms","jerk_max",
              "axis_std","logp_loco","logp_fog","fog_ratio","loco_frac"]
J = HORIZONS.index(2.0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
assert dev == "cuda", "GPU is OFF - enable the T4 accelerator before committing"

X   = np.load(PRE/"windows_X.npy")
M   = pd.read_parquet(PRE/"windows_meta.parquet")
DEV = np.load(PRE/"window_dev.npy")
B   = pd.read_parquet(PRE/"daily_baselines.parquet")
print("loaded:", X.shape, M.shape, DEV.shape, B.shape)


In [ ]:
# ---- rebuild the ablation cohort exactly as before (deterministic)
ABL   = M.index[M.has_baseline == 1].to_numpy()
Mab   = M.loc[ABL].reset_index(drop=True)
Xab   = X[ABL]; DEVab = DEV[ABL]
Yab   = Mab[[f"y_{h}" for h in HORIZONS]].to_numpy(np.float32)
NEGab = Mab["y_neg"].to_numpy(np.float32)

Mab["fold"] = -1
for k,(_,te) in enumerate(GroupKFold(n_splits=N_FOLDS).split(Mab, groups=Mab.subject)):
    Mab.loc[te,"fold"] = k
assert Mab.groupby("subject").fold.nunique().max() == 1, "SUBJECT LEAK"

# clinical state
subj = pd.read_csv(ROOT/"subjects.csv")
subj["sex_m"] = (subj.Sex.astype(str).str.upper().str[0]=="M").astype(float)
CLIN = ["Age","YearsSinceDx","UPDRSIII_On","UPDRSIII_Off","NFOGQ"]
C = Mab[["subject","visit"]].rename(columns={"subject":"Subject","visit":"Visit"}) \
      .merge(subj[["Subject","Visit","sex_m"]+CLIN], on=["Subject","Visit"], how="left")
C["med_on"] = (Mab.medication.astype(str).str.lower().str.startswith("on")).astype(float)
CLIN_COLS = CLIN + ["sex_m","med_on"]
for c in CLIN_COLS: C[f"{c}_miss"] = C[c].isna().astype(float)
CLINF = C[CLIN_COLS + [f"{c}_miss" for c in CLIN_COLS]].to_numpy(np.float32)

# subject-level personalisation vector
BCOLS = [f"base_{n}_mean" for n in FEAT_NAMES] + ["activity_fraction"]
BASE = np.nan_to_num(B.set_index("Subject")[BCOLS].reindex(Mab.subject).to_numpy(np.float32))

print("cohort:", Mab.subject.nunique(), "patients |", len(Mab), "windows")
print("clinical:", CLINF.shape, "| baseline:", BASE.shape)
print(Mab.groupby("fold").agg(windows=("subject","size"), subjects=("subject","nunique")))


In [ ]:
class Net(nn.Module):
    def __init__(self, n_side=0, nh=len(HORIZONS)):
        super().__init__()
        def blk(i,o,k,s): return nn.Sequential(
            nn.Conv1d(i,o,k,stride=s,padding=k//2), nn.BatchNorm1d(o), nn.GELU())
        self.enc = nn.Sequential(
            blk(3,32,7,2), blk(32,64,5,2), blk(64,96,5,2), blk(96,128,3,2),
            nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.side = nn.Sequential(nn.Linear(n_side,64), nn.GELU()) if n_side else None
        d = 128 + (64 if self.side else 0)
        self.head = nn.Sequential(nn.Linear(d,128), nn.GELU(), nn.Dropout(0.3), nn.Linear(128,nh))
    def forward(self, x, s=None):
        z = self.enc(x)
        if self.side is not None: z = torch.cat([z, self.side(s)], 1)
        return self.head(z)

def masked_bce(logits, y, neg, pos_w):
    """Head h learns ONLY from its own positives and the clean (>10 s) negatives."""
    m = (y > 0) | neg.unsqueeze(1).bool()
    l = nn.functional.binary_cross_entropy_with_logits(
        logits, y, pos_weight=pos_w, reduction="none")
    return (l*m).sum() / m.sum().clamp(min=1)

def run_seed(mode, seed, epochs=12, bs=256, lr=2e-3):
    """A = accel | B = accel + baseline + deviation | C = B + clinical"""
    torch.manual_seed(seed); np.random.seed(seed)
    use_pb, use_clin = mode in ("B","C"), mode == "C"
    rows, OOF = [], np.full((len(Mab), len(HORIZONS)), np.nan, np.float32)
    folds = Mab.fold.to_numpy()
    for f in range(N_FOLDS):
        tr, te = np.flatnonzero(folds != f), np.flatnonzero(folds == f)
        mu = Xab[tr].astype(np.float32).mean((0,1))
        sd = Xab[tr].astype(np.float32).std((0,1)) + 1e-6

        ptr, pte = [], []
        if use_pb:
            bm, bsd = BASE[tr].mean(0), BASE[tr].std(0) + 1e-6
            ptr += [(BASE[tr]-bm)/bsd, np.clip(DEVab[tr],-8,8)]
            pte += [(BASE[te]-bm)/bsd, np.clip(DEVab[te],-8,8)]
        if use_clin:
            cm = np.nan_to_num(np.nanmedian(CLINF[tr],0))
            cf = np.where(np.isnan(CLINF), cm, CLINF)
            cmu, csd = cf[tr].mean(0), cf[tr].std(0)+1e-6
            ptr.append((cf[tr]-cmu)/csd); pte.append((cf[te]-cmu)/csd)
        S_tr = np.concatenate(ptr,1).astype(np.float32) if ptr else None
        S_te = np.concatenate(pte,1).astype(np.float32) if pte else None

        pw = torch.tensor([max((NEGab[tr].sum()/max(Yab[tr,j].sum(),1))**0.5,1.0)
                           for j in range(len(HORIZONS))], dtype=torch.float32, device=dev)
        net = Net(n_side=S_tr.shape[1] if S_tr is not None else 0).to(dev)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*max(len(tr)//bs,1))

        for ep in range(epochs):
            net.train(); perm = np.random.permutation(len(tr))
            for i in range(0, len(perm)-bs+1, bs):
                k = perm[i:i+bs]; b = tr[k]
                xb = torch.from_numpy(((Xab[b].astype(np.float32)-mu)/sd).transpose(0,2,1)).to(dev)
                sb = torch.from_numpy(S_tr[k]).to(dev) if S_tr is not None else None
                opt.zero_grad()
                masked_bce(net(xb,sb), torch.from_numpy(Yab[b]).to(dev),
                           torch.from_numpy(NEGab[b]).to(dev), pw).backward()
                opt.step(); sch.step()

        net.eval(); P=[]
        with torch.no_grad():
            for i in range(0, len(te), 1024):
                b = te[i:i+1024]
                xb = torch.from_numpy(((Xab[b].astype(np.float32)-mu)/sd).transpose(0,2,1)).to(dev)
                sb = torch.from_numpy(S_te[i:i+1024]).to(dev) if S_te is not None else None
                P.append(torch.sigmoid(net(xb,sb)).cpu().numpy())
        P = np.concatenate(P); OOF[te] = P

        for j,h in enumerate(HORIZONS):
            m = (Yab[te,j]>0) | (NEGab[te]>0)
            if Yab[te,j][m].sum() < 5: continue
            rows.append(dict(mode=mode, seed=seed, fold=f, horizon=h,
                             n_pos=int(Yab[te,j][m].sum()),
                             prevalence=float(Yab[te,j][m].mean()),
                             auprc=average_precision_score(Yab[te,j][m], P[m,j])))
    print(f"  {mode} seed {seed} done")
    return pd.DataFrame(rows), OOF

SEEDS = [0,1,2]
res, oofs = [], {m: [] for m in ["A","B","C"]}
for s in SEEDS:
    for m in ["A","B","C"]:
        r, o = run_seed(m, s)
        res.append(r); oofs[m].append(o)

R3 = pd.concat(res, ignore_index=True)
R3.to_parquet(OUT/"ablation_seeds.parquet", index=False)
for m in ["A","B","C"]:
    np.save(OUT/f"oof_{m}_mean.npy", np.nanmean(np.stack(oofs[m]), axis=0))
print("saved ablation_seeds.parquet and seed-averaged OOF")


In [ ]:
# ============================================================
# Is the A/B/C difference real, or seed noise?
# ============================================================
per_seed = R3.groupby(["mode","seed"]).auprc.mean().unstack()
print("macro AUPRC by seed:\n", per_seed.round(4), "\n")
print("across seeds (mean / sd):\n",
      pd.DataFrame({"mean": per_seed.mean(1), "sd": per_seed.std(1)}).round(4), "\n")

print("horizon curve, averaged over seeds AND folds:")
print(R3.pivot_table(index="horizon", columns="mode", values="auprc").round(4))
print("\nchance:", R3.groupby("horizon").prevalence.mean().round(4).to_dict())

W = R3.pivot_table(index=["seed","fold","horizon"], columns="mode", values="auprc")
print("\npaired comparisons over", len(W), "(seed, fold, horizon) cells")
for a,b in [("C","A"), ("C","B"), ("B","A")]:
    d = (W[a]-W[b]).dropna()
    print(f"  {a} - {b}: mean {d.mean():+.4f}  sd {d.std():.4f}  "
          f"SEM {d.sem():.4f}  wins {int((d>0).sum())}/{len(d)}")

# seed-to-seed spread vs the effect being claimed
spread = per_seed.max(1) - per_seed.min(1)
print("\nseed-to-seed spread per mode:", spread.round(4).to_dict())
print("C-A effect:", round(float(W['C'].mean()-W['A'].mean()), 4))
print("\nVERDICT: claim 'C > A' only if the paired win rate is clearly above chance")
print("         AND the C-A effect exceeds the seed-to-seed spread.")

# ---- per-subject check on the seed-averaged model
OC = np.load(OUT/"oof_C_mean.npy"); OA = np.load(OUT/"oof_A_mean.npy")
rows=[]
for s,g in Mab.groupby("subject"):
    ix = g.index.to_numpy(); ix = ix[(Yab[ix,J]>0)|(NEGab[ix]>0)]
    if Yab[ix,J].sum() < 10: continue
    rows.append(dict(subject=s, prevalence=float(Yab[ix,J].mean()),
                     auprc_A=average_precision_score(Yab[ix,J], OA[ix,J]),
                     auprc_C=average_precision_score(Yab[ix,J], OC[ix,J])))
PS2 = pd.DataFrame(rows); PS2["gain"] = PS2.auprc_C - PS2.auprc_A
PS2.to_parquet(OUT/"per_subject_h2_seedavg.parquet", index=False)
print(f"\nwithin-subject at {HORIZONS[J]}s, n={len(PS2)}")
print(f"  A mean {PS2.auprc_A.mean():.4f} median {PS2.auprc_A.median():.4f}")
print(f"  C mean {PS2.auprc_C.mean():.4f} median {PS2.auprc_C.median():.4f}")
print(f"  C beats A in {int((PS2.gain>0).sum())}/{len(PS2)} patients")
print(f"  below 2x own prevalence (C): {int((PS2.auprc_C < 2*PS2.prevalence).sum())}/{len(PS2)}")
